In [1]:
import open3d as o3d
import numpy as np

point_cloud_file = "../outputs/export_splat_clean 3/3DGS_Result.ply"

pcd = o3d.io.read_point_cloud(str(point_cloud_file))
print(f"Points: {len(pcd.points)}, has normals: {pcd.has_normals()}")

# o3d.visualization.draw_geometries(
#     [pcd],
#     point_show_normal=False
# )

Points: 191983, has normals: True


In [2]:
# 1. Statistical Outlier Removal
# nb_neighbors: Number of neighbors to analyze for each point
# std_ratio: Standard deviation multiplier (lower = more aggressive filtering)
cl, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

# Extract inliers and outliers
inlier_cloud = pcd.select_by_index(ind)
outlier_cloud = pcd.select_by_index(ind, invert=True)

print(f"Original points: {len(pcd.points)}")
print(f"Inliers: {len(inlier_cloud.points)} | Outliers: {len(outlier_cloud.points)}")


Original points: 191983
Inliers: 185531 | Outliers: 6452


In [3]:
import numpy as np
import plotly.graph_objects as go

def plot_pcd_inline(cloud, max_points=60000, point_size=1.5):
    # Downsample if too large for browser rendering
    if len(cloud.points) > max_points:
        step = len(cloud.points) // max_points
        points = np.asarray(cloud.points)[::step]
        colors = np.asarray(cloud.colors)[::step] if cloud.has_colors() else None
    else:
        points = np.asarray(cloud.points)
        colors = np.asarray(cloud.colors) if cloud.has_colors() else None

    # Format RGB colors for Plotly if available
    marker_config = dict(size=point_size)
    if colors is not None and len(colors) > 0:
        marker_config['color'] = [f'rgb({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)})' for c in colors]
    else:
        marker_config['color'] = points[:, 2] # Color by Z-coordinate
        marker_config['colorscale'] = 'Viridis'

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:, 0],
                y=points[:, 1],
                z=points[:, 2],
                mode='markers',
                marker=marker_config
            )
        ]
    )

    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        height=600
    )
    fig.show()

# Run it in your cell:
plot_pcd_inline(pcd)


In [4]:
# Less aggressive filtering (preserves sparser surface regions)
filtered_pcd, ind = pcd.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=2.5  # Increase this (default in Open3D is typically 2.0 - 3.0)
)


# 1. Reconstruct a continuous mesh from the filtered point cloud
# depth controls resolution (8-10 is typical; higher = finer detail)
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    filtered_pcd, 
    depth=9
)

# 2. Trim low-density artifacts (Poisson creates a bounding bubble around the object)
densities = np.asarray(densities)
density_threshold = np.quantile(densities, 0.05) # Remove lowest 5% density (air/artifacts)
mesh.remove_vertices_by_mask(densities < density_threshold)

# 3. Sample points back from the filled mesh to get a complete, dense point cloud
filled_pcd = mesh.sample_points_poisson_disk(
    number_of_points=len(filtered_pcd.points)
)

print(f"Original: {len(pcd.points)} -> Filtered: {len(filtered_pcd.points)} -> Filled: {len(filled_pcd.points)}")


[WARNING] /Users/runner/work/Open3D/Open3D/build/poisson/src/ext_poisson/PoissonRecon/Src/FEMTree.Initialize.inl (Line 193)
          Initialize
          Found bad data: 187921


IndexError: index -1 is out of bounds for axis 0 with size 0

In [ ]:
# Extract numpy arrays
orig_pts = np.asarray(filtered_pcd.points)
fill_pts = np.asarray(filled_pcd.points)

# Subsample for rendering
orig_sample = orig_pts[::max(1, len(orig_pts) // 40000)]
fill_sample = fill_pts[::max(1, len(fill_pts) // 40000)]

fig = go.Figure(data=[
    go.Scatter3d(
        x=orig_sample[:, 0], y=orig_sample[:, 1], z=orig_sample[:, 2],
        mode='markers',
        name='Original Filtered PCD',
        marker=dict(size=1.5, color='lightgray', opacity=0.5)
    ),
    go.Scatter3d(
        x=fill_sample[:, 0], y=fill_sample[:, 1], z=fill_sample[:, 2],
        mode='markers',
        name='Filled Surface PCD',
        marker=dict(size=1.5, color='limegreen', opacity=0.7)
    )
])

fig.update_layout(
    title="Hole-Filling Comparison",
    scene=dict(aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40),
    height=650,
    legend=dict(x=0.02, y=0.98)
)
fig.show()


In [ ]:
# --- CONFIGURATION ---
MAX_POINTS = 60000  # Subsample limit for fluid browser 3D rendering
POINT_SIZE = 1.5

# 1. Extract coordinates and RGB colors
points = np.asarray(filled_pcd.points)
colors = np.asarray(filled_pcd.colors)  # Open3D stores RGB in [0.0, 1.0]

# 2. Subsample so browser rendering is fast and smooth
if len(points) > MAX_POINTS:
    step = len(points) // MAX_POINTS
    sample_points = points[::step]
    sample_colors = colors[::step]
else:
    sample_points = points
    sample_colors = colors

# 3. Convert [0.0, 1.0] float RGB to standard 'rgb(R, G, B)' strings for Plotly
rgb_uint8 = (sample_colors * 255).astype(np.uint8)
color_strings = [f'rgb({r},{g},{b})' for r, g, b in rgb_uint8]

# 4. Render with Plotly
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=sample_points[:, 0],
            y=sample_points[:, 1],
            z=sample_points[:, 2],
            mode='markers',
            marker=dict(
                size=POINT_SIZE,
                color=color_strings
            )
        )
    ]
)

fig.update_layout(
    title=f"Filled Point Cloud with Actual Colors ({len(sample_points):,} points displayed)",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=650
)

fig.show()


In [ ]:
output_ply_path = "../outputs/Dense_Fused_1_filled.ply"

# write_ascii=False saves in binary format (much smaller file size and faster to load)
success = o3d.io.write_point_cloud(
    output_ply_path, 
    filled_pcd, 
    write_ascii=False, 
    compressed=True
)

if success:
    print(f"Successfully saved {len(filled_pcd.points):,} points to: {output_ply_path}")
else:
    print("Failed to save the point cloud.")


Successfully saved 1,488,799 points to: ../outputs/Dense_Fused_1_filled.ply


In [ ]:
from pathlib import Path
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import pycolmap

# 1. Load the dense point cloud
dense_pcd = o3d.io.read_point_cloud("../outputs/Dense_Fused_1_filled.ply")
dense_pts = np.asarray(dense_pcd.points)
dense_colors = np.asarray(dense_pcd.colors)

# Subsample dense points for smooth browser rendering
step = max(1, len(dense_pts) // 50000)
dense_pts_sample = dense_pts[::step]
dense_colors_sample = (dense_colors[::step] * 255).astype(np.uint8)
dense_rgb_strings = [f"rgb({r},{g},{b})" for r, g, b in dense_colors_sample]

# 2. Extract the 258 camera positions from COLMAP
colmap_sparse_dir = Path("../data/colmap_output/multi_ring_1/sparse/0")
if not colmap_sparse_dir.exists():
    colmap_sparse_dir = Path("../data/colmap_output/multi_ring_1/sparse")

reconstruction = pycolmap.Reconstruction(str(colmap_sparse_dir))
camera_centers = np.array(
    [img.projection_center() for img in reconstruction.images.values()]
)

# 3. Create interactive 3D plot
fig = go.Figure()

# Layer 1: Dense Parasol Model
fig.add_trace(
    go.Scatter3d(
        x=dense_pts_sample[:, 0],
        y=dense_pts_sample[:, 1],
        z=dense_pts_sample[:, 2],
        mode="markers",
        name="Dense Parasol",
        marker=dict(size=1.5, color=dense_rgb_strings),
    )
)

# Layer 2: COLMAP Camera Poses (Orbits/Rings)
fig.add_trace(
    go.Scatter3d(
        x=camera_centers[:, 0],
        y=camera_centers[:, 1],
        z=camera_centers[:, 2],
        mode="markers+lines",
        name=f"Camera Trajectory ({len(camera_centers)} views)",
        marker=dict(size=3, color="crimson", symbol="diamond"),
        line=dict(color="rgba(255, 0, 0, 0.3)", width=2),
    )
)

fig.update_layout(
    title=f"COLMAP Multi-Ring Trajectory ({len(camera_centers)} cameras) + Dense 3D Model",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=700,
    legend=dict(x=0.02, y=0.98),
)

fig.show()


In [ ]:
# 1. Extract sparse 3D feature points & colors
sparse_pts = np.array([p.xyz for p in reconstruction.points3D.values()])
sparse_colors = np.array(
    [p.color / 255.0 for p in reconstruction.points3D.values()]
)

sparse_pcd = o3d.geometry.PointCloud()
sparse_pcd.points = o3d.utility.Vector3dVector(sparse_pts)
sparse_pcd.colors = o3d.utility.Vector3dVector(sparse_colors)

# 2. Filter sparse outliers
clean_sparse, _ = sparse_pcd.remove_statistical_outlier(
    nb_neighbors=20, std_ratio=2.0
)

# 3. Combine dense cloud + sparse features
integrated_pcd = dense_pcd + clean_sparse

print(f"Dense points: {len(dense_pcd.points):,}")
print(f"Sparse points added: {len(clean_sparse.points):,}")
print(f"Total integrated points: {len(integrated_pcd.points):,}")

# Export the combined point cloud
o3d.io.write_point_cloud(
    "../outputs/parasol_integrated.ply", integrated_pcd, write_ascii=False
)


Dense points: 1,488,799
Sparse points added: 48,862
Total integrated points: 1,537,661


True